In [1]:
import os
import torch

gpu_list = [0]
gpu_list_str = ','.join(map(str, gpu_list))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", gpu_list_str)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
import torch.nn as nn
import torchvision.models as models

class fully_connected(nn.Module):
    def __init__(self, model, num_ftrs, num_classes):  
        super(fully_connected, self).__init__()
        self.model = model
        self.fc_4 = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        x = self.model(x)
        x = torch.flatten(x, 1)
        out_1 = x
        out_3 = self.fc_4(x)
        out_3 = torch.relu(out_3)
        return out_1, out_3

In [3]:
from PIL import Image, ImageFile
import pandas as pd
from torchvision import transforms

class PTC_cell(torch.utils.data.Dataset):
    def __init__(self, root, slide='11_breast_cancer3',transform=None, stain_norm=False):
        super(PTC_cell, self).__init__()
        self.root = root
        self.slide = slide
        self.transform = transform
        self.stain_norm = stain_norm

        patch_path = os.path.join(root, slide, 'patches')
        patch = os.listdir(patch_path)
        patch_list = [x.split('.')[0] for x in patch]

        cell_label = pd.read_csv(os.path.join(root, slide, 'cell_ratio.csv'), index_col=0)
        gene_label = pd.read_csv(os.path.join(root, slide, 'high_250_stdata.csv'), index_col=0)
        label_df = pd.merge(gene_label, cell_label, left_index=True, right_index=True)

        label_index_set = set(label_df.index)
        patch_index_set = set(patch_list)
        and_set = label_index_set & patch_index_set

        patch_list = list(and_set)
        self.label_df = label_df.loc[patch_list]
        self.patch = patch_list
        
        self.coordinate_df = pd.read_csv(os.path.join(root, slide, 'spots.csv'), index_col=0)


    def __getitem__(self, index):
        patch_id = self.patch[index]
        patch_path = os.path.join(self.root, self.slide, 'patches', patch_id)
        patch = Image.open(patch_path+'.jpg').convert('RGB')
        data = transforms.Resize((224, 224))(patch)
        if self.transform is not None:
            data = self.transform(data)

        label = self.label_df.loc[patch_id].values
        label = torch.Tensor(label)
        
        slide = self.slide
        coordinate = self.coordinate_df.loc[patch_id].values

        return patch_id, data, label, slide, coordinate

    def __len__(self):
        return len(self.patch)

In [4]:
from glob import glob
tif_list = glob('/data1/r20user3/shared_project/Hist2Cell/code/training/train_test_splits/stnet/test*')
tif_list.sort()
test_cases = list()
for tif in tif_list:
    tif_path = tif.split('_')[-1].split('.')[0]
    test_cases.append(tif_path)
test_cases

['23209',
 '23268',
 '23269',
 '23270',
 '23272',
 '23277',
 '23287',
 '23288',
 '23377',
 '23450',
 '23506',
 '23508',
 '23567',
 '23803',
 '23810',
 '23895',
 '23901',
 '23903',
 '23944',
 '24044',
 '24105',
 '24220',
 '24223']

In [5]:
from tqdm import tqdm
import numpy as np
from scipy.stats import pearsonr
import joblib
from torchvision import transforms
import collections
import warnings

warnings.filterwarnings("ignore")

celltype_num = 39

model = models.densenet121(pretrained=True)
model.features = nn.Sequential(model.features, nn.AdaptiveAvgPool2d(output_size=(1, 1)))
num_ftrs = model.classifier.in_features
model_final = fully_connected(model.features, num_ftrs, celltype_num)   
checkpoint = torch.load("/data1/r20user3/shared_project/Hist2Cell/code/training/model_weights/breast_cross_source_epoch100_lr1e-4_2hop_densenet_onlycell_best_cell_all_abundance_average.pth")
new_state_dict = collections.OrderedDict()
for k, v in checkpoint.items():
    name = k[7:]  # remove "module."
    new_state_dict[name] = v
model2_dict = model_final.state_dict()
state_dict = {k:v for k,v in new_state_dict.items() if k in model2_dict.keys()}
model2_dict.update(state_dict)
model_final.load_state_dict(model2_dict)
model = model_final
model.to(device)
    
for case in test_cases:
    celltype_num = 39
    batch_size = 128
    patch_dir = "/data1/r20user3/shared_project/Hist2Cell/data/stnet"
    
    test_set = "/data1/r20user3/shared_project/Hist2Cell/code/training/train_test_splits/stnet/test_leave_"+case+".txt"
    test_slides = open(test_set).read().split('\n')
    
    test_transform_pcam = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    test_data_list = list()
    for slide in test_slides:
        test_data = PTC_cell(root=patch_dir, slide=slide,transform=test_transform_pcam)
        test_data_list.append(test_data)
    test_data = torch.utils.data.ConcatDataset(test_data_list)
    test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=8)
    
    with torch.no_grad():
        model.eval()

        test_cell_label_array = []
        test_cell_pred_array = []
        test_spot_id_array = []
        test_slide_id_array = []
        test_coordinate_array = []
        test_sample_num = 0
        for name, data, label, slide_id_list, coordinate_list in test_loader:
            data = data.to(device)
            label = label.to(device)
            label = label.float()
            label = label.squeeze()
            gene_label = label[:, :250]
            cell_abundance_label = label[:, 250:]
            test_cell_label_array.append(cell_abundance_label.cpu().detach().numpy())
            _, output = model(data)

            cell_abundance_output = output

            test_sample_num = test_sample_num + data.size(0)
            test_cell_pred_array.append(output.squeeze().cpu().detach().numpy())
            test_spot_id_array.append(name)
            test_slide_id_array.append(slide_id_list)
            test_coordinate_array.append(coordinate_list)

    test_cell_pred_array = np.concatenate(test_cell_pred_array)
    test_cell_label_array = np.concatenate(test_cell_label_array)
    
    merged_spot_id_list = [item for sublist in test_spot_id_array for item in sublist]
    merged_slide_id_list = [item for sublist in test_slide_id_array for item in sublist]
    merged_coordiante_list = np.concatenate(test_coordinate_array)
    
    test_cell_abundance_pos_pearson_average = 0.0
    test_cell_abundance_all_pearson_average = 0.0
    test_cell_abundance_pos_pearson_count = 0   
    test_cell_pearson_list = []
    for i in range(celltype_num):
        r, p = pearsonr(test_cell_pred_array[:, i], test_cell_label_array[:, i])
        if r > 0.0 and p <= 0.05:
            test_cell_abundance_pos_pearson_count = test_cell_abundance_pos_pearson_count + 1
            test_cell_abundance_pos_pearson_average = test_cell_abundance_pos_pearson_average + r
        if np.isnan(r):
            r = 0.0
        test_cell_abundance_all_pearson_average = test_cell_abundance_all_pearson_average + r
        test_cell_pearson_list.append(r)
    if test_cell_abundance_pos_pearson_count >= 1:
        test_cell_abundance_pos_pearson_average /= test_cell_abundance_pos_pearson_count
    else:
        test_cell_abundance_pos_pearson_average = 0.0
    test_cell_abundance_all_pearson_average = test_cell_abundance_all_pearson_average / celltype_num

    print(case)
    print("saving " + "best cell all abundance average " + str(test_cell_abundance_all_pearson_average))
    
    Predictions = dict()
    
    for slide_id in test_slides:
        indices = [index for index, value in enumerate(merged_slide_id_list) if value == slide_id]
        test_cell_pred_array_sub = test_cell_pred_array[indices]
        test_cell_label_array_sub = test_cell_label_array[indices]
        test_cell_pos_arraay_sub = merged_coordiante_list[indices]
        
        test_cell_abundance_pos_pearson_average = 0.0
        test_cell_abundance_all_pearson_average = 0.0
        test_cell_abundance_pos_pearson_count = 0   
        test_cell_pearson_list = []
        for i in range(celltype_num):
            r, p = pearsonr(test_cell_pred_array_sub[:, i], test_cell_label_array_sub[:, i])
            if r > 0.0 and p <= 0.05:
                test_cell_abundance_pos_pearson_count = test_cell_abundance_pos_pearson_count + 1
                test_cell_abundance_pos_pearson_average = test_cell_abundance_pos_pearson_average + r
            if np.isnan(r):
                r = 0.0
            test_cell_abundance_all_pearson_average = test_cell_abundance_all_pearson_average + r
            test_cell_pearson_list.append(r)
        if test_cell_abundance_pos_pearson_count >= 1:
            test_cell_abundance_pos_pearson_average /= test_cell_abundance_pos_pearson_count
        else:
            test_cell_abundance_pos_pearson_average = 0.0
        test_cell_abundance_all_pearson_average = test_cell_abundance_all_pearson_average / celltype_num
        
        Predictions[slide_id] = {
            'cell_abundance_predictions': test_cell_pred_array_sub,
            'cell_abundance_labels': test_cell_label_array_sub,
            'coords': test_cell_pos_arraay_sub,
        }
        
        print(slide_id) 
        print(test_cell_abundance_all_pearson_average)  
         
    # break  

    save_path = "/data1/r20user3/shared_project/Hist2Cell/code/analysis/inference/breast_cross_source/breast_cross_source_epoch100_lr1e-4_2hop_densenet_onlycell_"+case+"_best_cell_all_abundance_average.pkl"
    joblib.dump(Predictions, save_path)

23209
saving best cell all abundance average 0.2401339466810913
23209_C1
0.214132542263364
23209_C2
0.24138784337082436
23209_D1
0.2849096395264775
23268
saving best cell all abundance average 0.314183071026508
23268_C1
0.3002677917510769
23268_C2
0.34830820348829844
23268_D1
0.33605520163812197
23269
saving best cell all abundance average 0.13066847859108352
23269_C1
0.12349390253740569
23269_C2
0.12997678201146726
23269_D1
0.15067014625317543
23270
saving best cell all abundance average 0.03028627688745014
23270_D2
0.1513389130177522
23270_E1
0.08268072735081824
23270_E2
-0.09877537214426746
23272
saving best cell all abundance average 0.22413495045723267
23272_D2
0.25573750299496534
23272_E1
0.2584059048666099
23272_E2
0.15605017268111418
23277
saving best cell all abundance average 0.09298592981002726
23277_D2
0.11439991442982467
23277_E1
0.17969993054535502
23277_E2
-0.004706027428717194
23287
saving best cell all abundance average 0.43377313566031045
23287_C1
0.37478646257588377
